In [ ]:
from pathlib import Path

import pandas as pd


CWD = Path.cwd()
PROJECT_ROOT = CWD if (CWD / "_data").exists() else CWD.parents[2]

RAW_DIR = PROJECT_ROOT / "_data" / "01_raw"
OUTPUT_DIR = PROJECT_ROOT / "_data" / "02_interim" / "260429 name changed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COLUMN_RENAME_MAPS = {
    "Membership.csv": {
        "user_no": "USER_KEY",
        "product_cd": "product_code",
        "amount": "price",
        "concurrent_streams": "max_screen",
        "promotion_yn": "is_promotion",
        "repurchase": "is_repurchase",
    },
    "Movie_Master.csv": {
        "MOVIE_ID": "MOVIE_NUM",
        "TITLE": "movie_title",
        "RELEASE_MONTH": "ott_release_month",
    },
    "User_Mapping.csv": {
        "uid": "USER_KEY",
        "USER_ID": "USER_NUM",
    },
    "View_History.csv": {
        "USER_ID": "USER_NUM",
        "MOVIE_ID": "MOVIE_NUM",
        "DURATION": "watch_time(min)",
        "WATCH_DAY": "watch_day",
        "WATCH_SEQ": "watch_seq",
    },
}

membership = pd.read_csv(RAW_DIR / "Membership.csv")
membership = membership.rename(columns=COLUMN_RENAME_MAPS["Membership.csv"])
membership.to_csv(OUTPUT_DIR / "Membership.csv", index=False, encoding="utf-8", lineterminator="\n")

membership.head()

In [ ]:
movie_master = pd.read_csv(RAW_DIR / "Movie_Master.csv")
movie_master = movie_master.rename(columns=COLUMN_RENAME_MAPS["Movie_Master.csv"])
movie_master.to_csv(OUTPUT_DIR / "Movie_Master.csv", index=False, encoding="utf-8", lineterminator="\n")

movie_master.head()

In [ ]:
user_mapping = pd.read_csv(RAW_DIR / "User_Mapping.csv")
user_mapping = user_mapping.rename(columns=COLUMN_RENAME_MAPS["User_Mapping.csv"])
user_mapping.to_csv(OUTPUT_DIR / "User_Mapping.csv", index=False, encoding="utf-8", lineterminator="\n")

user_mapping.head()

In [ ]:
view_history = pd.read_csv(RAW_DIR / "View_History.csv")
view_history = view_history.rename(columns=COLUMN_RENAME_MAPS["View_History.csv"])
view_history.to_csv(OUTPUT_DIR / "View_History.csv", index=False, encoding="utf-8", lineterminator="\n")

view_history.head()

In [ ]:
description = pd.read_excel(RAW_DIR / "Description.xlsx", sheet_name="Spec")
description = description.copy()

membership_rename_map = COLUMN_RENAME_MAPS["Membership.csv"]
description["Name"] = description["Name"].replace(membership_rename_map)

description.to_excel(OUTPUT_DIR / "Description.xlsx", sheet_name="Spec", index=False)

description.head()

In [ ]:
rename_summary = []

for file_name, rename_map in COLUMN_RENAME_MAPS.items():
    raw_columns = pd.read_csv(RAW_DIR / file_name, nrows=0).columns.tolist()
    changed_columns = pd.read_csv(OUTPUT_DIR / file_name, nrows=0).columns.tolist()

    for before, after in zip(raw_columns, changed_columns):
        rename_summary.append(
            {
                "file": file_name,
                "before": before,
                "after": after,
                "changed": before != after,
            }
        )

raw_description = pd.read_excel(RAW_DIR / "Description.xlsx", sheet_name="Spec")
changed_description = pd.read_excel(OUTPUT_DIR / "Description.xlsx", sheet_name="Spec")

for before, after in zip(raw_description["Name"], changed_description["Name"]):
    rename_summary.append(
        {
            "file": "Description.xlsx",
            "before": before,
            "after": after,
            "changed": before != after,
        }
    )

rename_summary_df = pd.DataFrame(rename_summary)
rename_summary_df